<div style="border: 3px solid #b42318; background: #fef3f2; color: #7a271a; border-radius: 12px; padding: 16px 20px; line-height: 1.5">
<div style="font-size: 20px; font-weight: 800; color: #b42318; margin-bottom: 10px">
&#9888;&#65039; ЧЕРНОВИК &mdash; НЕ АКТУАЛЬНАЯ ВЕРСИЯ
</div>
<p style="margin: 0 0 10px 0"><b>Это занятие ещё в работе и будет переписано.</b>
Формулировки, данные и порядок заданий изменятся; часть материала может
опираться на то, что к этому моменту курса ещё не прочитано.</p>
<p style="margin: 0">Заниматься по нему пока не нужно &mdash; дождитесь
окончательной версии. Актуально сейчас только <b>занятие&nbsp;1</b>.</p>
</div>

# Лабораторная работа 5. Переобучение, смещение–разброс и скользящий контроль

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| Место в курсе | после лекции 4 |
| Опора на лекции | лекция 4: переобучение (опр. 4.1), разложение смещение–разброс (теорема 4.3), отложенная выборка и $q$-кратный скользящий контроль (опр. 4.7–4.8), LOO, принцип структурной минимизации риска (опр. 4.17) |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Это методологическое занятие курса: всё, что делается дальше, опирается на протокол оценки качества, который выстраивается здесь. Разложим ошибку на смещение и разброс численно, увидим на числах, сколько стоит нарушение правил валидации, и соберём честный протокол выбора модели.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab05_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Данные занятия — учебные, одинаковые у всех.** Числа на экране у преподавателя
> и у вас совпадают, поэтому можно сверяться с соседом и спорить о результате вслух.
> Индивидуальная таблица, порождённая по вашему ФИО, появится в домашней работе.
>
> Почти все учебные выборки курса берутся из `sklearn.datasets` — это либо
> готовые наборы, либо порождённые генератором:
> [7.1. Игрушечные наборы](https://scikit-learn.ru/stable/datasets/toy_dataset.html) ·
> [7.2. Реальные наборы](https://scikit-learn.ru/stable/datasets/real_world.html) ·
> [7.3. Генераторы выборок](https://scikit-learn.ru/stable/datasets/sample_generators.html)

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import (GridSearchCV, KFold, ShuffleSplit,
                                     StratifiedKFold, cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Полиномы высоких степеней плохо обусловлены (занятие 2) -- предупреждение
# ожидаемо и забивает вывод.
warnings.filterwarnings("ignore", message=".*Ill-conditioned matrix.*")

---
# Часть 1. Две кривые, которые надо уметь читать

Определение 4.1: переобучение — когда $Q(a,X^\ell)$ мал, а $R(a)$ велик.
Диагностируют это двумя графиками:

* **кривая сложности** — ошибка на обучении и на контроле как функция параметра
  сложности модели;
* **кривая обучения** — те же две ошибки как функция размера выборки $\ell$.

In [ ]:
F_TRUE = lambda x: np.sin(3 * x) + 0.4 * x ** 2
SIGMA = 0.35


def make_sample(n, generator):
    x = generator.uniform(-2, 2, n)
    return x[:, None], F_TRUE(x) + generator.normal(0, SIGMA, n)


fit_poly = lambda d: make_pipeline(PolynomialFeatures(d), Ridge(alpha=1e-8))

gen = np.random.default_rng(RANDOM_STATE)
X_tr, y_tr = make_sample(60, gen)
X_te, y_te = make_sample(3000, gen)
print(f"неустранимый шум: sigma^2 = {SIGMA ** 2:.4f}")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

degrees = [1, 2, 3, 5, 7, 9, 12, 15, 18]
rows = []
for d in degrees:
    m = fit_poly(d).fit(X_tr, y_tr)
    # TODO (2 строки): ошибка на обучающей (X_tr, y_tr) и на контрольной (X_te, y_te)
    rows.append({"степень": d, "Q обучающая": ..., "Q контрольная": ...})
curve = pd.DataFrame(rows).set_index("степень")
display(curve.round(4))

In [ ]:
sizes = [15, 20, 30, 45, 70, 110, 180, 300, 500]
tr_err, te_err = [], []
for n in sizes:
    e1, e2 = [], []
    for rep in range(20):
        g = np.random.default_rng(1000 + rep)
        Xs, ys = make_sample(n, g)
        m = fit_poly(9).fit(Xs, ys)
        e1.append(np.mean((m.predict(Xs) - ys) ** 2))
        e2.append(np.mean((m.predict(X_te) - y_te) ** 2))
    tr_err.append(np.mean(e1)); te_err.append(np.mean(e2))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(curve.index, curve["Q обучающая"], "o-", lw=2, label="обучение")
ax1.plot(curve.index, curve["Q контрольная"], "s-", lw=2, label="контроль")
ax1.axhline(SIGMA ** 2, ls="--", color="black", label=r"$\sigma^2$")
ax1.set_yscale("log"); ax1.set_xlabel("степень полинома"); ax1.set_ylabel("$Q$")
ax1.set_title("Кривая сложности"); ax1.legend(fontsize=8)

ax2.plot(sizes, tr_err, "o-", lw=2, label="обучение")
ax2.plot(sizes, te_err, "s-", lw=2, label="контроль")
ax2.axhline(SIGMA ** 2, ls="--", color="black", label=r"$\sigma^2$")
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_xlabel(r"размер выборки $\ell$"); ax2.set_ylabel("$Q$")
ax2.set_title("Кривая обучения (степень 9)"); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

> **Вывод.** К какому уровню сходятся обе кривые обучения и почему именно к нему? Что означает большой зазор между кривыми и что — их схождение на высоком уровне?
>
> *(ваш ответ здесь)*

---
# Часть 2. Разложение смещение–разброс: считаем всё три слагаемых

Теорема 4.3:

$$
\mathbb{E}_{X^\ell,\varepsilon}\bigl[(y - a(x))^2\bigr]
= \underbrace{\bigl(f(x) - \mathbb{E}_{X^\ell}[a(x)]\bigr)^2}_{\text{смещение}^2}
+ \underbrace{\mathrm{Var}_{X^\ell}\bigl(a(x)\bigr)}_{\text{разброс}}
+ \underbrace{\sigma^2}_{\text{шум}} .
$$

На реальных данных эти слагаемые вычислить нельзя — $f$ и распределение выборки
неизвестны. Но здесь **мы сами породили данные**, поэтому можем посчитать всё
и проверить равенство.

In [ ]:
N_REP, N_TRAIN = 300, 60
x_eval = np.linspace(-2, 2, 200)
f_eval = F_TRUE(x_eval)

preds_by_degree = {}
for d in [1, 3, 5, 9, 15]:
    preds = np.empty((N_REP, len(x_eval)))
    for r in range(N_REP):
        g = np.random.default_rng(10_000 + r)        # СВОЯ выборка в каждом повторении
        Xs, ys = make_sample(N_TRAIN, g)
        preds[r] = fit_poly(d).fit(Xs, ys).predict(x_eval[:, None])
    preds_by_degree[d] = preds
print(f"обучено моделей: {len(preds_by_degree)} степеней x {N_REP} повторений")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

rows = []
for d, preds in preds_by_degree.items():
    # preds -- матрица (N_REP, 200): предсказания всех повторений на сетке x_eval
    # TODO (2 строки):
    #   смещение^2 = среднее по сетке от (f_eval - среднее предсказание)^2
    #   разброс    = среднее по сетке от дисперсии предсказаний по повторениям
    bias2, var = ..., ...
    rows.append({"степень": d, "смещение^2": bias2, "разброс": var,
                 "шум": SIGMA ** 2, "сумма": bias2 + var + SIGMA ** 2})
bv = pd.DataFrame(rows).set_index("степень")
display(bv.round(4))

In [ ]:
# Сверка: измеряем E[(y - a(x))^2] напрямую и сравниваем с суммой трёх слагаемых
gen_check = np.random.default_rng(777)
bv["измеренная ошибка"] = [
    np.mean((f_eval[None, :] + gen_check.normal(0, SIGMA, preds.shape) - preds) ** 2)
    for preds in preds_by_degree.values()]
bv["расхождение, %"] = (bv["измеренная ошибка"] / bv["сумма"] - 1) * 100
display(bv[["сумма", "измеренная ошибка", "расхождение, %"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, d in zip(axes, [1, 5, 15]):
    preds = preds_by_degree[d]
    for r in range(30):
        ax.plot(x_eval, preds[r], color="#128C7E", alpha=0.18, lw=0.9)
    ax.plot(x_eval, preds.mean(axis=0), color="#C97A2B", lw=2.5, label=r"$\bar a$")
    ax.plot(x_eval, f_eval, "k--", lw=2, label="$f$")
    b2 = np.mean((f_eval - preds.mean(axis=0)) ** 2); v = np.mean(preds.var(axis=0))
    ax.set_title(fr"степень {d}: смещение$^2$ = {b2:.3f}, разброс = {v:.3f}", fontsize=10)
    ax.set_xlabel("$x$"); ax.set_ylim(f_eval.min() - 2.5, f_eval.max() + 2.5)
axes[0].set_ylabel("$y$"); axes[0].legend()
plt.tight_layout(); plt.show()

> **Вывод.** Совпала ли сумма трёх слагаемых с измеренной ошибкой? Опишите словами, что видно на «облаке» предсказаний в каждом из трёх случаев.
>
> *(ваш ответ здесь)*

---
# Часть 3. Утечка при валидации: сколько она стоит

Скользящий контроль честен, **только если** ни один объект контрольного блока
не повлиял на обучение — включая предобработку и отбор признаков.

Самая наглядная проверка — на **чистом шуме**: данные, в которых заведомо нет
никакой связи с целью. Честная оценка обязана дать AUC $\approx 0.5$.
Всё, что выше, — артефакт протокола, а не свойство данных.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import roc_auc_score

g = np.random.default_rng(RANDOM_STATE)
X_noise = g.normal(size=(120, 4000))          # 4000 признаков чистого шума
y_noise = g.integers(0, 2, 120)               # метки не связаны с признаками
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print("в этих данных связи нет по построению: истинное AUC = 0.5")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

# TODO: посчитайте две оценки AUC по 5-кратному стратифицированному контролю.
#   (а) НЕПРАВИЛЬНО: SelectKBest(f_classif, k=20) применён ко ВСЕЙ выборке,
#       и только потом идёт разбиение на блоки;
#   (б) ПРАВИЛЬНО: отбор -- шаг Pipeline, то есть выполняется заново
#       внутри каждого блока (используйте cross_val_score).
# Истинное значение для этих данных: AUC = 0.5.

> **Вывод.** Какое AUC дала утечка на чистом шуме и откуда оно взялось? Почему масштабирование «протекает» слабее, чем отбор признаков?
>
> *(ваш ответ здесь)*

---
# Часть 4. Какую схему контроля выбрать

Определение 4.8: выборка делится на $q$ блоков, каждый по очереди служит
контролем, и оценки усредняются. Схему выбирают не по вкусу, а по свойствам
самой **оценки**: она случайна, и у неё есть смещение относительно истинного
риска и дисперсия.

> **Напоминание — осторожно: «смещение» здесь другое.** В части 2 смещение и разброс были свойствами **модели**: насколько её средний
> прогноз отличается от истинной зависимости и насколько он пляшет от выборки
> к выборке.
>
> Здесь те же два слова описывают свойства **числа, которым мы оцениваем
> качество**. Оценка $\widehat R$ случайна (зависит от того, как выборка
> разбилась на блоки), и у неё есть своё смещение
> $\mathbb{E}\widehat R - R$ и своя дисперсия. Например, при $q=2$ модель
> учится на половине данных, поэтому получается хуже, чем на полной выборке, —
> оценка систематически **занижает** качество, и это смещение оценки, а не модели.
>
> Термины совпали случайно; на что смотрим — на модель или на оценку —
> приходится держать в голове самому.
>
> LOO (leave-one-out) — предельный случай $q = \ell$: каждый объект по очереди
> остаётся единственным контрольным.

In [ ]:
from sklearn.model_selection import LeaveOneOut
import time

L, N_EXP = 80, 100
X_big, y_big = make_sample(20_000, np.random.default_rng(4242))
schemes = {
    "отложенная 25%": lambda n: list(ShuffleSplit(1, test_size=.25, random_state=0)
                                     .split(np.zeros(n))),
    "5-кратный": lambda n: list(KFold(5, shuffle=True, random_state=0).split(np.zeros(n))),
    "10-кратный": lambda n: list(KFold(10, shuffle=True, random_state=0).split(np.zeros(n))),
    "LOO": lambda n: list(LeaveOneOut().split(np.zeros(n))),
}

In [ ]:
records, cost = [], {nm: 0.0 for nm in schemes}
for e in range(N_EXP):
    g = np.random.default_rng(50_000 + e)
    Xs, ys = make_sample(L, g)
    row = {"истинный риск": np.mean((fit_poly(5).fit(Xs, ys).predict(X_big) - y_big) ** 2)}
    for nm, splitter in schemes.items():
        t0 = time.perf_counter()
        row[nm] = float(np.mean([
            np.mean((fit_poly(5).fit(Xs[tr], ys[tr]).predict(Xs[te]) - ys[te]) ** 2)
            for tr, te in splitter(L)]))
        cost[nm] += time.perf_counter() - t0
    records.append(row)

res = pd.DataFrame(records)
err = res[list(schemes)].sub(res["истинный риск"], axis=0)
summary = pd.DataFrame({
    "смещение": err.mean(),
    "станд. ошибка смещения": err.std() / np.sqrt(N_EXP),
    "разброс ошибки оценивания": err.std(),
    "обучений на оценку": pd.Series({"отложенная 25%": 1, "5-кратный": 5,
                                     "10-кратный": 10, "LOO": L}),
    "время, мс": pd.Series({k: v / N_EXP * 1000 for k, v in cost.items()}),
})
display(summary.round(4))

> **Вывод.** У каких схем смещение значимо отличается от нуля и в какую сторону? Почему на практике берут $q = 5$–$10$, а не LOO?
>
> *(ваш ответ здесь)*

---
# Часть 5. Честный протокол целиком

Собираем всё вместе на `breast_cancer` — той самой задаче, которую занятие 4
оставило с гиперпараметрами по умолчанию и обещанием «корректный протокол будет
в занятии 5». Предобработка внутри `Pipeline`, гиперпараметры подбираются
скользящим контролем по обучающей выборке, а контрольная не участвует ни в чём
до самого конца.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

Xc, yc = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(
    Xc, yc, test_size=0.25, random_state=RANDOM_STATE, stratify=yc)

# Масштабирование ВНУТРИ Pipeline: иначе на каждом фолде оно подсмотрит
# отложенную часть этого фолда -- ровно та утечка, что в части 3.
pipe = Pipeline([("scale", StandardScaler()),
                 ("model", LogisticRegression(max_iter=5000))])
cv5 = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
gs = GridSearchCV(pipe, {"model__C": np.logspace(-3, 3, 13)},
                  cv=cv5, scoring="roc_auc").fit(Xtr, ytr)

auc_test = roc_auc_score(yte, gs.predict_proba(Xte)[:, 1])
print(f"лучшее C по скользящему контролю: {gs.best_params_['model__C']:.4g}")
print(f"оценка CV                       : {gs.best_score_:.4f}")
print(f"AUC на контроле (честно)        : {auc_test:.4f}")
print(f"разрыв CV - контроль            : {gs.best_score_ - auc_test:+.4f}")

### Смещение есть, но по одному разбиению его не увидеть

Разрыв получился почти нулевым, а то и отрицательным. Значит ли это, что
«максимум по сетке» — честная оценка? Нет: смещение — величина **средняя**, и
на одном разбиении она тонет в шуме. Повторим эксперимент по 20 разбиениям при
разных объёмах обучающей выборки.

In [ ]:
def cv_minus_test(n_train, seed):
    Xa, Xb, ya, yb = train_test_split(Xc, yc, test_size=0.25,
                                      random_state=seed, stratify=yc)
    Xa, ya = Xa[:n_train], ya[:n_train]
    g = GridSearchCV(pipe, {"model__C": np.logspace(-3, 3, 13)},
                     cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                     scoring="roc_auc").fit(Xa, ya)
    return g.best_score_ - roc_auc_score(yb, g.predict_proba(Xb)[:, 1])


rows = []
for n_train in (40, 60, 100, 200, len(ytr)):
    gaps = np.array([cv_minus_test(n_train, s) for s in range(20)])
    rows.append({"обучающих": n_train, "средний разрыв": gaps.mean(),
                 "ст. ошибка": gaps.std(ddof=1) / np.sqrt(len(gaps))})
display(pd.DataFrame(rows).set_index("обучающих").round(4))

> **Вывод.** Почему «лучшее значение по сетке» — уже не честная оценка качества? Почему на полной выборке разрыв оказался отрицательным, и что показывает усреднение?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Модель A: смещение большое, разброс малый. Модель B: наоборот. Какая выиграет при $\ell = 50$ и какая при $\ell = 50\,000$? Почему?
2. Почему оценка по отложенной выборке систематически пессимистична, а «лучшее значение по сетке» — оптимистично? Это одно и то же смещение?
3. Коллега отобрал 50 признаков из 10 000 по корреляции с целью на всех данных, затем честно провёл 10-кратный контроль и получил AUC 0.85. Что вы ему скажете и как перепроверите?
4. Вы получили CV AUC = 0.91 и AUC на отложенной выборке = 0.86. Это норма или признак ошибки? Какие три причины проверите первыми?

---

**Дома:** откройте `lab05_homework.ipynb` — там три задачи: свой скользящий контроль, вложенный контроль и VC-размерность перебором разметок.